# Аналитика продаж магазина электроники — задание 1.2

Этот notebook реализует **только аналитическую часть задания 1.2** на синтетическом наборе из **10 000 строк и 18 столбцов**. Одна строка описывает одну товарную позицию внутри заказа.

Логика работы: **цель и аналитические вопросы → подготовка окружения → загрузка → первичный осмотр → контроль качества → очистка/ETL → preprocessing → EDA с визуализациями и описательными статистиками → проверка гипотез → сохранение очищенных данных → контроль результата**.

**Граница комплекта:** построение дашбордов, графиков, Story и публикация в **Yandex DataLens** относятся к следующему заданию **2.1** и в этом notebook не выполняются. Результатом задания 1.2 являются очищенные и подготовленные датасеты, которые затем можно использовать как источник для BI.

## 1. Цель анализа

Цель — подготовить качественные данные о продажах учебного магазина электроники и провести исследовательский анализ, чтобы получить проверенный набор данных для последующего использования в BI.

На этапе 1.2 мы:

- проверяем качество исходных данных;
- выполняем очистку и преобразование данных (ETL);
- создаём необходимые аналитические признаки (preprocessing);
- проводим EDA с описательными статистиками и графиками;
- формулируем выводы и ограничения;
- сохраняем очищенный и аналитически подготовленный CSV.

**Дашборд Yandex DataLens на этом этапе не строится.**

## 2. Аналитические вопросы и гипотезы

Основные вопросы для EDA:

1. Как меняются выручка, прибыль и число заказов по месяцам?
2. Какие категории, бренды, города и каналы формируют основной результат?
3. В каких категориях выше доля возвратов и средняя скидка?
4. Какие товары формируют наибольшую выручку?
5. Какие числовые признаки связаны между собой и насколько сильна эта связь?

Рабочие гипотезы:

- **H1:** распределение скидки в онлайн-каналах отличается от офлайн-магазина;
- **H2:** категория товара связана с фактом возврата;
- **H3:** между размером скидки и количеством единиц в строке есть монотонная связь.

Гипотеза — это проверяемое предположение, а не заранее известный ответ. Статистическая значимость сама по себе не доказывает причинность и должна интерпретироваться вместе с величиной эффекта и контекстом.

## 3. Данные, единица наблюдения и ожидаемый результат

Источник — синтетический CSV `data/raw/electronics_sales_10000.csv`.

**Grain / единица наблюдения:** одна товарная позиция внутри заказа. Поэтому `line_id` должен быть уникальным после удаления полных дублей, а `order_id` может повторяться.

В нашем кейсе исходные данные представлены **одной таблицей**, поэтому отдельная консолидация нескольких таблиц не требуется. Если бы факты продаж, товары и клиенты хранились раздельно, перед очисткой потребовалась бы проверяемая консолидация по ключам.

Ожидаемый результат задания 1.2:

- `data/processed/electronics_sales_clean.csv` — очищенные исходные поля;
- `data/processed/electronics_sales_analysis_ready.csv` — очищенные данные + производные аналитические признаки;
- графики EDA в `outputs/eda_charts/`;
- описательные статистики, аналитические выводы и проверка рабочих гипотез;
- `RESULTS_SUMMARY.md` — краткая итоговая сводка.

**Не входит в задание 1.2:** создание чартов и дашбордов Yandex DataLens, Story, публикация и настройка BI-приложения. Это отдельный этап 2.1.

## 4. Подготовка окружения

`pandas` используется для таблиц, `NumPy` — для числовых расчётов, `matplotlib` — для графиков, `SciPy` — для статистических тестов. Пути задаются относительно корня проекта, чтобы notebook не зависел от компьютера автора.

В этом разделе также создаются только каталоги, относящиеся к заданию 1.2: `data/processed` и `outputs/eda_charts`.

In [ ]:
# Импортируем Path для кроссплатформенной работы с путями.
from pathlib import Path
# Импортируем warnings для управления некритичными предупреждениями.
import warnings
# Импортируем NumPy для числовых расчётов.
import numpy as np
# Импортируем pandas для работы с табличными данными.
import pandas as pd
# Импортируем matplotlib для построения графиков EDA.
import matplotlib.pyplot as plt
# Импортируем статистические тесты из SciPy.
from scipy.stats import mannwhitneyu, chi2_contingency, spearmanr
# Импортируем display и Markdown для аккуратного вывода в notebook.
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (11, 6)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

PROJECT_DIR = Path('.')
RAW_PATH = PROJECT_DIR / 'data' / 'raw' / 'electronics_sales_10000.csv'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
CLEAN_PATH = PROCESSED_DIR / 'electronics_sales_clean.csv'
ANALYSIS_READY_PATH = PROCESSED_DIR / 'electronics_sales_analysis_ready.csv'
CHART_DIR = PROJECT_DIR / 'outputs' / 'eda_charts'
SUMMARY_PATH = PROJECT_DIR / 'RESULTS_SUMMARY.md'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

print('Исходный файл:', RAW_PATH)
print('Очищенный файл:', CLEAN_PATH)
print('Подготовленный для анализа файл:', ANALYSIS_READY_PATH)
print('Каталог графиков EDA:', CHART_DIR)

## 5. Загрузка данных

`pd.read_csv()` читает CSV в `DataFrame`. Сразу проверяем, что файл существует и содержит ожидаемые **10 000 строк**.

In [ ]:
# TODO: выполните раздел «5. Загрузка данных» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 6. Словарь полей

Ключевой момент: `line_id` идентифицирует строку продажи, а `order_id` — заказ. Один заказ может содержать несколько товаров, поэтому для количества заказов нельзя использовать обычное число строк.

In [ ]:
# TODO: выполните раздел «6. Словарь полей» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 7. Первичный осмотр данных

Проверяем типы, пропуски, описательную статистику и число уникальных значений. На этом этапе ничего не исправляем: сначала фиксируем симптомы качества.

In [ ]:
# TODO: выполните раздел «7. Первичный осмотр данных» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 8. Проверка типов и преобразование даты

Дата в исходном файле намеренно записана в нескольких текстовых форматах. `format='mixed'` позволяет разобрать смешанные представления, а `errors='coerce'` делает нераспознанные даты явными пропусками.

In [ ]:
# TODO: выполните раздел «8. Проверка типов и преобразование даты» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 9. Пропущенные значения

Пропуск не равен нулю. Сначала измеряем количество и долю пропусков, затем применяем разные правила в зависимости от смысла поля.

In [ ]:
# TODO: выполните раздел «9. Пропущенные значения» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 10. Дубликаты и уникальность идентификаторов

Полный дубль завышает строковые суммы. Проверяем и полные дубликаты, и повторяемость `line_id`. Повторяющийся `order_id` является нормой, потому что один заказ может включать несколько товарных позиций.

In [ ]:
# TODO: выполните раздел «10. Дубликаты и уникальность идентификаторов» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 11. Согласованность категориальных значений

Пробелы и регистр могут искусственно создать отдельные категории. Нормализуем строки через `strip()` и `casefold()`, затем возвращаем утверждённые русские названия.

In [ ]:
# TODO: выполните раздел «11. Согласованность категориальных значений» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 12. Числовые правила и поиск аномалий

Для учебного примера задаём проверяемые диапазоны:

- `quantity`: 1–10;
- `unit_price_rub`: 300–500 000 руб.;
- `discount_pct`: 0–0,60;
- `unit_cost_rub`: 100–400 000 руб.

Это **правила конкретного синтетического кейса**, а не универсальные границы для реального ритейла.

In [ ]:
# TODO: выполните раздел «12. Числовые правила и поиск аномалий» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 13. Очистка данных (ETL)

Удаляем полные дубликаты, восстанавливаем пропуски и аномалии устойчивыми правилами. Для цен/себестоимости используем медиану по товару, для скидки — медиану по каналу и категории, для количества — медиану по категории. Категориальные пропуски восстанавливаем из устойчивого контекста.

После очистки сохраняем **отдельный базовый очищенный CSV**. Он содержит только исходные 18 полей, но уже без намеренно заложенных дефектов качества.

In [ ]:
# TODO: выполните раздел «13. Очистка данных (ETL)» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 14. Контроль качества после очистки

Очистка завершена только после повторной проверки. `assert` делает требования исполняемыми: при нарушении notebook остановится до аналитических выводов.

In [ ]:
# TODO: выполните раздел «14. Контроль качества после очистки» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 15. Preprocessing и производные признаки

На этапе preprocessing создаём признаки, которые упрощают дальнейший EDA и будут полезны на следующем BI-этапе. При этом мы **не строим BI-дашборд и не создаём DataLens-специфичные витрины**.

Возврат в этом учебном кейсе упрощённо обнуляет выручку и реализованную себестоимость строки; это модельное допущение, а не бухгалтерское правило.

In [ ]:
# TODO: выполните раздел «15. Preprocessing и производные признаки» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 16. Общие KPI продаж

Считаем показатели на правильной гранулярности. Количество заказов — это `nunique(order_id)`, а средний чек — общая чистая выручка / число уникальных заказов.

In [ ]:
# TODO: выполните раздел «16. Общие KPI продаж» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 17. Распределения и выбросы после очистки

После очистки выбросы могут оставаться как реальные редкие значения. Поэтому визуально проверяем распределения цены, выручки строки и скидки, не удаляя значения автоматически только из-за их редкости.

In [ ]:
# TODO: выполните раздел «17. Распределения и выбросы после очистки» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 18. Временная динамика

Агрегируем данные по месяцам: выручка, прибыль, уникальные заказы и возвраты. Для полного года это удобный базовый уровень анализа сезонности.

In [ ]:
# TODO: выполните раздел «18. Временная динамика» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 19. Категории, бренды, каналы и города

Сравниваем не только выручку, но и прибыль, маржинальность и возвраты. Это защищает от вывода «лидер по выручке = лучший сегмент».

In [ ]:
# TODO: выполните раздел «19. Категории, бренды, каналы и города» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 20. Топ товаров и брендов

Топ по выручке помогает выделить товары, которые формируют наибольший вклад в результат. На этапе 1.2 это часть EDA; более специализированные методы сегментации товарного портфеля относятся к отдельным задачам и здесь не выполняются.

In [ ]:
# TODO: выполните раздел «20. Топ товаров и брендов» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 21. Возвраты и скидки

Возврат — риск для выручки и клиентского опыта. Скидка может стимулировать объём, но её эффект нельзя оценивать только по росту количества: важны прибыль и состав заказов.

In [ ]:
# TODO: выполните раздел «21. Возвраты и скидки» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 22. Корреляционный анализ

Используем коэффициент Спирмена для монотонных связей между числовыми признаками. Корреляция показывает связь, а не причинность.

In [ ]:
# TODO: выполните раздел «22. Корреляционный анализ» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 23. Статистическая проверка гипотез

Проверяем три гипотезы. Уровень значимости для учебного примера: `alpha = 0.05`.

- H1: Mann–Whitney U — сравнение распределений скидки у офлайн и онлайн строк;
- H2: χ² Пирсона — связь категории и факта возврата;
- H3: Spearman — монотонная связь скидки и количества.

In [ ]:
# TODO: выполните раздел «23. Статистическая проверка гипотез» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 24. Сохранение результатов задания 1.2

К этому моменту сформированы два результата:

1. `electronics_sales_clean.csv` — очищенная версия исходной таблицы;
2. `electronics_sales_analysis_ready.csv` — та же очищенная выборка с производными аналитическими признаками.

Второй файл является удобным входом для следующего задания 2.1, где уже отдельно выполняется загрузка в Yandex DataLens и построение BI-визуализаций.

In [ ]:
# TODO: выполните раздел «24. Сохранение результатов задания 1.2» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

## 25. Что передаётся в следующее задание 2.1

Задание 1.2 заканчивается **подготовленными данными**, а не BI-дашбордом.

Основной файл для следующего этапа:

```text
data/processed/electronics_sales_analysis_ready.csv
```

Он содержит строковый уровень продаж и производные показатели. На следующем этапе отдельно решается, какие KPI, чарты, селекторы и дополнительные витрины нужны в Yandex DataLens.

Эти действия относятся к **заданию 2.1** и намеренно не включены в текущий notebook.

## 26. Финальный QA задания 1.2

Перед завершением проверяем, что очищенные файлы действительно созданы, размеры согласованы, критические дефекты устранены и EDA сформировал ожидаемые графики.

In [ ]:
# TODO: выполните раздел «26. Финальный QA задания 1.2» самостоятельно.
# Сверяйтесь с описанием над ячейкой и фиксируйте промежуточные проверки.
# Не переходите к следующему шагу, пока не можете объяснить полученный результат.

### Источники по инструментам и методам

- pandas: https://pandas.pydata.org/docs/
- NumPy: https://numpy.org/doc/
- Matplotlib: https://matplotlib.org/stable/
- SciPy statistics: https://docs.scipy.org/doc/scipy/reference/stats.html
- CRISP-DM overview, указанный в формулировке задания: https://www.sv-europe.com/crisp-dm-methodology/

Yandex DataLens намеренно не используется внутри задания 1.2: BI-визуализации и дашборды относятся к следующему заданию 2.1.